# Create the full transcripts for each meeting
Combine the segments in the correct order

In [97]:
import pandas as pd
import numpy as np
import json
import os
import re
from pathlib import Path
import xml.etree.ElementTree as ET
NITE = "http://nite.sourceforge.net/" # Global attribute used for ID in all xml files...

## Create a CSV file for all timestamped segments
This should serve as a way to compute WER based on timestamp alignment. It takes the predicted transcript segments and gather the segments from the ground truth transcriptions that correlate with the timestamps from the ASR output.
The .csv file should have the following information:
* segment_id
* participant_id
* meeting_id
* start
* end
* text


### First get the metadata for all the meetings and participants for the AMI Meeting Corpus:
This will be useful when analysing the speaker-wise WER at a later point..

The info_metadata.csv contains the following metadata:
- meeting_id
- meeting
- participant_id
- channel
- sex
- age
- language

While the segment_metadata.csv contains the following:
- segment_id
- start (time)
- end (time)
- meeting_id
- channel

In [134]:
resource_path = os.path.join('../', 'data/amicorpus/metadata/corpusResources')
meetings_root = ET.parse(os.path.join(resource_path, 'meetings.xml')).getroot()
participants_root = ET.parse(os.path.join(resource_path, 'participants.xml')).getroot()

# Pattern to match on either: [ES2002-ES2016], [IS1000-IS1009], [TS3003-3012]
pattern = re.compile(
    r"^(ES20(0[4]|1[4])|IS100[9]|TS30(0[3]|0[7]))"
)

meetings = []
participants = []


for meeting in meetings_root.findall(".//meeting"):
    meeting_id = meeting.attrib["observation"]
    if not pattern.match(meeting_id[:-1]):
        continue
    
    for speaker in meeting.findall(".//speaker"):
        meetings.append({
            "meeting_id": meeting_id,
            'meeting': meeting_id[:-1],
            'participant_id': speaker.attrib['global_name'],
            'channel': speaker.attrib['channel'] # represents the channel of the specific participant 
        })

meetings_df = pd.DataFrame(meetings)


for participant in participants_root.findall(".//participant"):
    participants.append(participant.attrib)

participants_df = pd.DataFrame(participants)
participants_df = participants_df.rename(columns={
    participants_df.columns[0]: 'participant_id'
})
info_df = meetings_df.merge(
    right=participants_df,
    how='inner',
    on=['meeting', 'participant_id']
)

meetings_df = meetings_df.drop(columns=['participant_id', 'channel'])
meetings_df.drop_duplicates(inplace=True)

This shows there is also participant metadata available for the ESXXX files.

In [133]:
participants_df = participants_df[participants_df['meeting'].str.match(pattern)]
participants_df.head(15)

,participant_id,sex,age_at_collection,native_language,meeting
8,FEE013,F,20.0,English,ES2004
9,MEE014,M,18.0,English,ES2004
10,MEO015,M,30.0,Hindi,ES2004
11,FEE016,F,38.0,English,ES2004
48,MEE053,M,47.0,English,ES2014
49,MEE054,M,19.0,English,ES2014
50,FEE055,F,22.0,English,ES2014
51,MEE056,M,26.0,English,ES2014


In [115]:
rows = []
words_path = os.path.join('../', 'data/amicorpus/metadata/words')

words = {}
word_order = []

# Pattern to match on either: [ES2002-ES2016], [IS1000-IS1009], [TS3003-3012]
pattern = re.compile(
    r"^(ES20(0[4]|1[4])|IS100[9]|TS30(0[3]|0[7])).*\.words\.xml$"
)


for xml_file in Path(words_path).glob("*.words.xml"):
    if not pattern.match(xml_file.name):
        continue

    root = ET.parse(xml_file).getroot()

    for word in root:
        w_id = word.attrib.get(f"{{{NITE}}}id")
        words[w_id] = {
            "start": float(word.attrib.get("starttime", "NaN")),
            "end": float(word.attrib.get("endtime", "NaN")),
            "word": word.text if word.text else 'NaN',
            "punc": word.attrib.get("punc")
        }
        word_order.append(w_id)

word_pos = {wid: i for i, wid in enumerate(word_order)}

In [116]:
rows = []
segments_path = os.path.join('../', 'data/amicorpus/metadata/segments')
utterances = []

# Pattern to match on either: [ES2002-ES2016], [IS1000-IS1009], [TS3003-3012]
pattern = re.compile(
    r"^(ES20(0[4]|1[4])|IS100[9]|TS30(0[3]|0[7])).*\.segments\.xml$"
)

for xml_file in Path(segments_path).glob("*.segments.xml"):
    if not pattern.match(xml_file.name):
        continue

    seg_root = ET.parse(xml_file).getroot()

    for seg in seg_root.findall('.//segment'):
        child = seg.find('.//{*}child')
        if child is None:
            continue

        ref = child.attrib['href']
        
        info = ref.split('#')[0]
        meeting_id = info.split('.')[0]
        channel = info.split('.')[1]

        word_ids = ref.split('#')[1]
        id_count = word_ids.count('id')
        matches = None
        if id_count > 1:
            # A range of words
            matches = re.search(r'id\(([^)]+)\)\.\.id\(([^)]+)\)', word_ids)
        else:
            # A single word
            matches = re.search(r'id\(([^)]+)\)', word_ids)

        word_group = [group for group in matches.groups()]
        utter_words = []
        if len(word_group) > 1:
            start = word_pos[word_group[0]]
            end = word_pos[word_group[1]]
            utter_words = word_order[start:end + 1]
        else:
            start = word_pos[word_group[0]]
            utter_words = word_order[start:start + 1]
        
        sentence = ''
        for id in utter_words:
            w = words[id]
            if w and w['word'] != 'NaN':
                if type(w['punc']) == str:
                    sentence = sentence + w['word']
                else:
                    sentence = sentence + ' ' + w['word']

        utterances.append({
        "segment_id": seg.attrib.get(f"{{{NITE}}}id"),
        "start": float(seg.attrib["transcriber_start"]),
        "end": float(seg.attrib["transcriber_end"]),
        "text": sentence,
        'meeting_id': meeting_id,
        'channel': seg.attrib['channel']
    })

utterances_df = pd.DataFrame(utterances)


Add path to the meeting dataframe before saving as a csv file

In [137]:
map = {

}

for root, dirs, files in os.walk('../data/amicorpus/test_split'):
    if len(files) == 1:
        folder = root.split('/')[-1]
        filepath = os.path.join(folder, files[0])
        # Add to dict map: 
        map[folder] = filepath


meetings_df['path'] = ' '

for i, row in meetings_df.iterrows():
    m_id = row['meeting_id']
    path = map.get(m_id)
    meetings_df.at[i, 'path'] = path

In [138]:
meetings_df.head(10)

,meeting_id,meeting,path
0,IS1009a,IS1009,IS1009a/IS1009a.Mix-Headset.wav
4,IS1009b,IS1009,IS1009b/IS1009b.Mix-Headset.wav
8,IS1009c,IS1009,IS1009c/IS1009c.Mix-Headset.wav
12,IS1009d,IS1009,IS1009d/IS1009d.Mix-Headset.wav
16,ES2004a,ES2004,ES2004a/ES2004a.Mix-Headset.wav
20,ES2004b,ES2004,ES2004b/ES2004b.Mix-Headset.wav
24,ES2004c,ES2004,ES2004c/ES2004c.Mix-Headset.wav
28,ES2004d,ES2004,ES2004d/ES2004d.Mix-Headset.wav
32,ES2014a,ES2014,ES2014a/ES2014a.Mix-Headset.wav
36,ES2014b,ES2014,ES2014b/ES2014b.Mix-Headset.wav


In [139]:
utterances_df.to_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/metadata/segment_metadata.csv', index=False)
meetings_df.to_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/metadata/meeting_metadata.csv', index=False)
participants_df.to_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/metadata/participant_metadata.csv', index=False)

## Align on timestamp
Take the ground truth segments and align them with the predicted segments on the segments start and end times.
First check to see how long the predicted segments are, and then check how much of the ground truth segments match within that same timeframe. If any segments contain overlapping speech, take those segments out and compute WER separately based on each speaker to see which speaker is best detected by the model. 

In [146]:
def fetch_results(path):    
    ids = []
    text = []
    start_time = []
    end_time = []
    paths = []



    with open(path, 'r') as file:
        lines = file.read().splitlines()

        for line in lines:
            json_line = json.loads(line)
            id = list(json_line)[0]

            for item in json_line.values():
                segment = item['segments']
                
                if len(segment) == 0:
                    ids.append(id)
                    paths.append(item['path'])
                    text.append('NaN')
                    start_time.append(np.nan)
                    end_time.append(np.nan)
                else:
                    for s in segment:
                        ids.append(id)
                        paths.append(item['path'])
                        text.append(s['text'])
                        start_time.append(s['start'])
                        end_time.append(s['end'])


    segments = pd.DataFrame()
    segments['id'] = ids
    segments['filename'] = paths
    segments['text'] = text
    segments['start'] = start_time
    segments['end'] = end_time

    return segments



In [163]:
segment_metadata = pd.read_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/metadata/segment_metadata.csv')
print(segment_metadata.shape)
segment_metadata.head(10)

(10172, 6)


,segment_id,start,end,text,meeting_id,channel
0,IS1009a.sync.326,55.680,56.102,Okay.,IS1009a,3
1,IS1009a.sync.328,57.232,57.744,I think so.,IS1009a,3
2,IS1009a.sync.330,61.312,62.637,"Yeah, that's a good plan.",IS1009a,3
3,IS1009a.sync.332,83.024,86.976,And I'm a marketing person. I wanna figure ou...,IS1009a,3
4,IS1009a.sync.334,88.688,89.728,My name is Eileen.,IS1009a,3
5,IS1009a.sync.336,118.736,119.120,Okay.,IS1009a,3
6,IS1009a.sync.338,131.104,131.568,Mm-hmm.,IS1009a,3
7,IS1009a.sync.340,145.520,145.856,No.,IS1009a,3
8,IS1009a.sync.342,159.472,160.057,Mm-hmm.,IS1009a,3
9,IS1009a.sync.344,162.992,163.392,NaN,IS1009a,3


In [173]:
segment_metadata = segment_metadata.dropna(subset=['text'])
segment_metadata['meeting'] = segment_metadata['meeting_id'].str[:-1]
print(segment_metadata.shape)

(9107, 7)


In [176]:
segment_metadata = segment_metadata.sort_values(by=['meeting_id', 'start'], ascending=True)

In [177]:
es2004a = segment_metadata[segment_metadata['meeting_id'].str.contains('ES2004a')]
es2004a.head(10)

,segment_id,start,end,text,meeting_id,channel,meeting
3522,ES2004a.sync.3,0.000,1.792,Hmm hmm hmm.,ES2004a,0,ES2004
9452,ES2004a.sync.101,10.944,14.737,Are we we're not allowed to dim the lights so...,ES2004a,1,ES2004
3523,ES2004a.sync.5,17.619,18.391,Yeah.,ES2004a,0,ES2004
9453,ES2004a.sync.103,18.722,20.320,"Okay, that's fine.",ES2004a,1,ES2004
9454,ES2004a.sync.105,22.352,23.872,Am I supposed to be standing up there?,ES2004a,1,ES2004
1447,ES2004a.sync.399,24.992,27.024,So we've got both of these clipped on?,ES2004a,3,ES2004
9455,ES2004a.sync.107,25.163,25.980,Okay.,ES2004a,1,ES2004
1448,ES2004a.sync.401,28.560,30.590,She gonna answer me or not?,ES2004a,3,ES2004
9456,ES2004a.sync.109,29.098,32.297,"Yeah, I've got",ES2004a,1,ES2004
1449,ES2004a.sync.403,31.280,33.480,"Right, both of them, okay.",ES2004a,3,ES2004


In [178]:
es2004b = segment_metadata[segment_metadata['meeting_id'].str.contains('ES2004b')]
es2004b.head(10)

,segment_id,start,end,text,meeting_id,channel,meeting
4254,ES2004b.sync.200,0.000,5.143,Help.,ES2004b,1,ES2004
4255,ES2004b.sync.202,8.164,11.017,It's up there? That screen's black.,ES2004b,1,ES2004
4256,ES2004b.sync.204,36.741,40.164,"Alright, okay. Okay, that's fine.",ES2004b,1,ES2004
9320,ES2004b.sync.718,47.685,49.284,Oh God.,ES2004b,3,ES2004
4257,ES2004b.sync.206,55.336,56.573,Are we done?,ES2004b,1,ES2004
4258,ES2004b.sync.208,66.784,79.270,"Right, okay um, this is our second meeting an...",ES2004b,1,ES2004
9321,ES2004b.sync.720,80.016,82.320,"Uh, no that's okay, sorry.",ES2004b,3,ES2004
4259,ES2004b.sync.210,81.245,83.536,"Okay, um",ES2004b,1,ES2004
4260,ES2004b.sync.212,85.856,88.743,"I'll go over what we decided last meeting,",ES2004b,1,ES2004
7797,ES2004b.sync.3,89.104,89.553,Mm-hmm.,ES2004b,0,ES2004


In [179]:
es2004c = segment_metadata[segment_metadata['meeting_id'].str.contains('ES2004c')]
es2004c.head(10)

,segment_id,start,end,text,meeting_id,channel,meeting
8111,ES2004c.sync.215,0.000,2.806,I'll wait until you're all um hooked up.,ES2004c,1,ES2004
5129,ES2004c.sync.813,17.379,19.197,Oh good grief.,ES2004c,3,ES2004
5130,ES2004c.sync.815,22.689,23.295,'Kay.,ES2004c,3,ES2004
8112,ES2004c.sync.217,29.611,30.520,Okay.,ES2004c,1,ES2004
5131,ES2004c.sync.817,49.827,50.685,Oh.,ES2004c,3,ES2004
8114,ES2004c.sync.221,52.113,54.976,Put it on in that way. Thanks.,ES2004c,1,ES2004
7324,ES2004c.sync.5,53.980,54.502,Oops.,ES2004c,0,ES2004
8115,ES2004c.sync.223,78.048,78.592,Okay.,ES2004c,1,ES2004
7325,ES2004c.sync.7,79.184,79.648,Mm.,ES2004c,0,ES2004
8116,ES2004c.sync.225,79.728,83.904,"Welcome back everybody, hope you've had fun.",ES2004c,1,ES2004


In [180]:
es2004d = segment_metadata[segment_metadata['meeting_id'].str.contains('ES2004d')]
es2004d.head(10)

,segment_id,start,end,text,meeting_id,channel,meeting
180,ES2004d.sync.314,3.434,6.896,If you leave them on the whole time you get t...,ES2004d,1,ES2004
8498,ES2004d.sync.3,13.008,14.192,Hmm.,ES2004d,0,ES2004
3729,ES2004d.sync.1034,15.088,16.176,Is that someone's?,ES2004d,3,ES2004
181,ES2004d.sync.316,19.787,20.906,Is that.,ES2004d,1,ES2004
8499,ES2004d.sync.5,22.797,23.737,Thank you.,ES2004d,0,ES2004
182,ES2004d.sync.318,23.936,25.296,"three, apparently.",ES2004d,1,ES2004
8500,ES2004d.sync.7,25.456,26.880,Hmm. Hmm.,ES2004d,0,ES2004
183,ES2004d.sync.320,27.360,28.704,"Okay, you all switched on.",ES2004d,1,ES2004
3730,ES2004d.sync.1036,27.754,28.643,Okay.,ES2004d,3,ES2004
8501,ES2004d.sync.9,28.928,29.680,"Yep, me too.",ES2004d,0,ES2004


In [181]:
es2014a = segment_metadata[segment_metadata['meeting_id'].str.contains('ES2014a')]
es2014a.head(10)

,segment_id,start,end,text,meeting_id,channel,meeting
5862,ES2014a.sync.4,71.376,72.160,"Right, so",ES2014a,0,ES2014
5863,ES2014a.sync.6,73.972,77.920,start of the first meeting. Uh.,ES2014a,0,ES2014
7137,ES2014a.sync.342,76.112,76.432,Mm-hmm.,ES2014a,3,ES2014
5864,ES2014a.sync.8,81.314,86.848,"Right, so agenda of the first meeting. Where ...",ES2014a,0,ES2014
5865,ES2014a.sync.10,88.912,104.595,We have twenty five minutes for this meeting....,ES2014a,0,ES2014
7000,ES2014a.sync.145,91.856,92.576,Okay.,ES2014a,1,ES2014
7138,ES2014a.sync.344,102.854,103.558,Yeah.,ES2014a,3,ES2014
7139,ES2014a.sync.346,104.832,106.816,I'm Robin. I'm the Marketing Manager.,ES2014a,3,ES2014
7691,ES2014a.sync.263,107.616,110.160,I'm Louisa. I'm the User Interface Designer.,ES2014a,2,ES2014
7001,ES2014a.sync.147,110.384,113.008,I'm Nick. I am the Industrial Designer.,ES2014a,1,ES2014


In [182]:
es2014b = segment_metadata[segment_metadata['meeting_id'].str.contains('ES2014b')]
es2014b.head(10)

,segment_id,start,end,text,meeting_id,channel,meeting
3088,ES2014b.sync.4,57.098,59.200,Right uh.,ES2014b,0,ES2014
3089,ES2014b.sync.6,62.383,64.416,So um.,ES2014b,0,ES2014
3090,ES2014b.sync.8,69.856,73.760,So where's the PowerPoint presentation?,ES2014b,0,ES2014
3091,ES2014b.sync.10,76.224,78.448,Sorry?,ES2014b,0,ES2014
3092,ES2014b.sync.12,81.216,83.472,"Microsoft PowerPoint, right.",ES2014b,0,ES2014
3093,ES2014b.sync.14,88.509,92.320,"Right, okay. So.",ES2014b,0,ES2014
3094,ES2014b.sync.16,97.543,98.208,Right.,ES2014b,0,ES2014
3095,ES2014b.sync.18,100.272,103.552,"Okay, so we've got uh",ES2014b,0,ES2014
3096,ES2014b.sync.20,107.568,112.304,so we've got new project requirements. Um.,ES2014b,0,ES2014
3097,ES2014b.sync.22,116.176,125.520,"So basically we've got three things, and we'v...",ES2014b,0,ES2014


In [183]:
es2014c = segment_metadata[segment_metadata['meeting_id'].str.contains('ES2014c')]
es2014c.head(10)

,segment_id,start,end,text,meeting_id,channel,meeting
7191,ES2014c.sync.3,90.975,91.936,Okay.,ES2014c,0,ES2014
7192,ES2014c.sync.5,112.135,113.244,Right.,ES2014c,0,ES2014
7193,ES2014c.sync.7,117.920,126.976,"Conceptual design meeting. Right. Okay, so",ES2014c,0,ES2014
7194,ES2014c.sync.9,128.736,142.208,Right well um from the last meeting I was try...,ES2014c,0,ES2014
7195,ES2014c.sync.11,143.360,146.848,"quick summary of the last uh meeting, I can",ES2014c,0,ES2014
7196,ES2014c.sync.13,148.064,149.504,quickly give you,ES2014c,0,ES2014
7197,ES2014c.sync.15,152.720,159.600,"what we what we had. Uh right, so",ES2014c,0,ES2014
7198,ES2014c.sync.17,165.274,182.656,Wishing I hadn't closed the damn Right so we ...,ES2014c,0,ES2014
7199,ES2014c.sync.19,183.808,186.816,made our decisions about uh,ES2014c,0,ES2014
7200,ES2014c.sync.21,189.952,192.976,"the device itself, that it was gonna be simpl...",ES2014c,0,ES2014


In [184]:
es2014d = segment_metadata[segment_metadata['meeting_id'].str.contains('ES2014d')]
es2014d.head(10)

,segment_id,start,end,text,meeting_id,channel,meeting
7895,ES2014d.sync.3,0.000,3.292,So is Why not save that.,ES2014d,0,ES2014
3963,ES2014d.sync.965,3.292,6.806,"No, you'll ha have to open it up from elsewhere.",ES2014d,3,ES2014
7896,ES2014d.sync.5,9.766,12.688,"Do you want to replace existing file, no.",ES2014d,0,ES2014
7898,ES2014d.sync.9,18.560,21.888,I actually tried to transfer it to My Documen...,ES2014d,0,ES2014
3964,ES2014d.sync.967,21.808,25.680,"Yeah, you have to you have to close that wind...",ES2014d,3,ES2014
3965,ES2014d.sync.969,27.004,28.113,And then find it.,ES2014d,3,ES2014
7899,ES2014d.sync.11,33.378,34.969,spreadsheet.,ES2014d,0,ES2014
7900,ES2014d.sync.13,47.056,54.133,"Yeah, but I've ta uh right, I'll just re-do i...",ES2014d,0,ES2014
7902,ES2014d.sync.17,71.350,72.050,Right.,ES2014d,0,ES2014
1704,ES2014d.sync.752,76.432,78.400,Well we've made our prototype anyway.,ES2014d,2,ES2014


In [185]:
is1009a = segment_metadata[segment_metadata['meeting_id'].str.contains('IS1009a')]
is1009a.head(10)

,segment_id,start,end,text,meeting_id,channel,meeting
1534,IS1009a.sync.4,54.928,60.880,Okay. Everybody ready? Uh I think the first t...,IS1009a,0,IS1009
0,IS1009a.sync.326,55.680,56.102,Okay.,IS1009a,3,IS1009
7643,IS1009a.sync.229,56.864,57.088,Yeah.,IS1009a,2,IS1009
1,IS1009a.sync.328,57.232,57.744,I think so.,IS1009a,3,IS1009
2,IS1009a.sync.330,61.312,62.637,"Yeah, that's a good plan.",IS1009a,3,IS1009
1535,IS1009a.sync.6,61.984,67.328,and everybody's name and what your function i...,IS1009a,0,IS1009
1810,IS1009a.sync.154,66.608,80.880,"Okay. Yeah, my name is Francina. And I'm uh a...",IS1009a,1,IS1009
1536,IS1009a.sync.8,76.048,76.528,Mm-hmm.,IS1009a,0,IS1009
1537,IS1009a.sync.10,81.008,82.192,Mm-hmm. Okay.,IS1009a,0,IS1009
3,IS1009a.sync.332,83.024,86.976,And I'm a marketing person. I wanna figure ou...,IS1009a,3,IS1009


In [170]:
e2004 = segment_metadata[segment_metadata['meeting_id'].str.contains('ES2004')]
e2004.head(10)

,segment_id,start,end,text,meeting_id,channel
3522,ES2004a.sync.3,0.000,1.792,Hmm hmm hmm.,ES2004a,0
9452,ES2004a.sync.101,10.944,14.737,Are we we're not allowed to dim the lights so...,ES2004a,1
3523,ES2004a.sync.5,17.619,18.391,Yeah.,ES2004a,0
9453,ES2004a.sync.103,18.722,20.320,"Okay, that's fine.",ES2004a,1
9454,ES2004a.sync.105,22.352,23.872,Am I supposed to be standing up there?,ES2004a,1
1447,ES2004a.sync.399,24.992,27.024,So we've got both of these clipped on?,ES2004a,3
9455,ES2004a.sync.107,25.163,25.980,Okay.,ES2004a,1
1448,ES2004a.sync.401,28.560,30.590,She gonna answer me or not?,ES2004a,3
9456,ES2004a.sync.109,29.098,32.297,"Yeah, I've got",ES2004a,1
1449,ES2004a.sync.403,31.280,33.480,"Right, both of them, okay.",ES2004a,3


In [171]:
e2014 = segment_metadata[segment_metadata['meeting_id'].str.contains('ES2014')]
e2014.head(10)

,segment_id,start,end,text,meeting_id,channel
5862,ES2014a.sync.4,71.376,72.160,"Right, so",ES2014a,0
5863,ES2014a.sync.6,73.972,77.920,start of the first meeting. Uh.,ES2014a,0
7137,ES2014a.sync.342,76.112,76.432,Mm-hmm.,ES2014a,3
5864,ES2014a.sync.8,81.314,86.848,"Right, so agenda of the first meeting. Where ...",ES2014a,0
5865,ES2014a.sync.10,88.912,104.595,We have twenty five minutes for this meeting....,ES2014a,0
7000,ES2014a.sync.145,91.856,92.576,Okay.,ES2014a,1
7138,ES2014a.sync.344,102.854,103.558,Yeah.,ES2014a,3
7139,ES2014a.sync.346,104.832,106.816,I'm Robin. I'm the Marketing Manager.,ES2014a,3
7691,ES2014a.sync.263,107.616,110.160,I'm Louisa. I'm the User Interface Designer.,ES2014a,2
7001,ES2014a.sync.147,110.384,113.008,I'm Nick. I am the Industrial Designer.,ES2014a,1


In [169]:
transcripts = segment_metadata.groupby('meeting_id')['text'].sum()
transcripts.head(10)

meeting_id
ES2004a     Hmm hmm hmm. Are we we're not allowed to dim ...
ES2004b     Help. It's up there? That screen's black. Alr...
ES2004c     I'll wait until you're all um hooked up. Oh g...
ES2004d     If you leave them on the whole time you get t...
ES2014a     Right, so start of the first meeting. Uh. Mm-...
ES2014b     Right uh. So um. So where's the PowerPoint pr...
ES2014c     Okay. Right. Conceptual design meeting. Right...
ES2014d     So is Why not save that. No, you'll ha have t...
IS1009a     Okay. Everybody ready? Uh I think the first t...
IS1009b     Okay, is everybody ready? Yeah? Yeah I'd to j...
Name: text, dtype: str

In [159]:
test0 = segment_metadata[(segment_metadata['meeting_id'] == 'IS1009a') & (segment_metadata['channel'] == 0)]
test0.head(10)

,segment_id,start,end,text,meeting_id,channel
1534,IS1009a.sync.4,54.928,60.880,Okay. Everybody ready? Uh I think the first t...,IS1009a,0
1535,IS1009a.sync.6,61.984,67.328,and everybody's name and what your function i...,IS1009a,0
1536,IS1009a.sync.8,76.048,76.528,Mm-hmm.,IS1009a,0
1537,IS1009a.sync.10,81.008,82.192,Mm-hmm. Okay.,IS1009a,0
1538,IS1009a.sync.12,87.424,88.624,Mm-hmm. And your name is?,IS1009a,0
1539,IS1009a.sync.14,90.160,90.576,Okay.,IS1009a,0
1540,IS1009a.sync.16,107.856,118.208,Very good. And as you already know I am Betty...,IS1009a,0
1541,IS1009a.sync.18,118.757,119.984,Um.,IS1009a,0
1542,IS1009a.sync.20,121.344,132.845,"Yes y opening, acquaintance, tool training we...",IS1009a,0
1543,IS1009a.sync.21,132.845,146.800,Uh we get ins each of us will get instruction...,IS1009a,0


In [160]:
test1 = segment_metadata[(segment_metadata['meeting_id'] == 'IS1009a') & (segment_metadata['channel'] == 1)]
test1.head(10)

,segment_id,start,end,text,meeting_id,channel
1810,IS1009a.sync.154,66.608,80.880,"Okay. Yeah, my name is Francina. And I'm uh a...",IS1009a,1
1813,IS1009a.sync.160,322.208,325.360,"Yes, I'm Francina. Yes, sure.",IS1009a,1
1814,IS1009a.sync.162,331.488,333.456,"No, Okay.",IS1009a,1
1815,IS1009a.sync.164,336.336,340.064,What should I draw?,IS1009a,1
1816,IS1009a.sync.166,347.968,350.080,I'm going to draw a snake.,IS1009a,1
1817,IS1009a.sync.168,355.792,359.062,How does it look like?,IS1009a,1
1819,IS1009a.sync.172,429.711,430.149,Yes.,IS1009a,1
1820,IS1009a.sync.174,453.920,454.426,"Yeah, I",IS1009a,1
1821,IS1009a.sync.176,461.440,486.768,"Yes, I I feel that all the remote should be v...",IS1009a,1
1822,IS1009a.sync.178,488.443,489.493,"Yes, exactly",IS1009a,1


In [161]:
test2 = segment_metadata[(segment_metadata['meeting_id'] == 'IS1009a') & (segment_metadata['channel'] == 2)]
test2.head(10)

,segment_id,start,end,text,meeting_id,channel
7643,IS1009a.sync.229,56.864,57.088,Yeah.,IS1009a,2
7644,IS1009a.sync.231,91.182,107.504,Yeah. Uh I'm Jeanne-Oui. Um uh my role is ind...,IS1009a,2
7650,IS1009a.sync.243,347.072,348.547,Snake.,IS1009a,2
7656,IS1009a.sync.255,428.672,431.822,"Yeah, of course, using remote control. Yeah.",IS1009a,2
7657,IS1009a.sync.257,450.531,451.822,Uh.,IS1009a,2
7658,IS1009a.sync.259,459.840,462.072,Yeah. Yeah.,IS1009a,2
7659,IS1009a.sync.261,481.473,484.514,Audio player. Oh. Okay.,IS1009a,2
7660,IS1009a.sync.263,490.157,490.583,Hmm.,IS1009a,2
7661,IS1009a.sync.265,492.784,495.568,Divides us Yeah. Yeah.,IS1009a,2
7663,IS1009a.sync.269,506.160,506.867,Yeah.,IS1009a,2


In [162]:
test3 = segment_metadata[(segment_metadata['meeting_id'] == 'IS1009a') & (segment_metadata['channel'] == 3)]
test3.head(10)

,segment_id,start,end,text,meeting_id,channel
0,IS1009a.sync.326,55.680,56.102,Okay.,IS1009a,3
1,IS1009a.sync.328,57.232,57.744,I think so.,IS1009a,3
2,IS1009a.sync.330,61.312,62.637,"Yeah, that's a good plan.",IS1009a,3
3,IS1009a.sync.332,83.024,86.976,And I'm a marketing person. I wanna figure ou...,IS1009a,3
4,IS1009a.sync.334,88.688,89.728,My name is Eileen.,IS1009a,3
5,IS1009a.sync.336,118.736,119.120,Okay.,IS1009a,3
6,IS1009a.sync.338,131.104,131.568,Mm-hmm.,IS1009a,3
7,IS1009a.sync.340,145.520,145.856,No.,IS1009a,3
8,IS1009a.sync.342,159.472,160.057,Mm-hmm.,IS1009a,3
11,IS1009a.sync.348,188.016,188.448,Okay.,IS1009a,3


In [141]:
participant_metadata = pd.read_csv('/root/master_thesis/thesis_multi_speaker_asr/data/amicorpus/metadata/participant_metadata.csv')
print(participant_metadata.shape)
participant_metadata.head(10)

(189, 5)


,participant_id,sex,age_at_collection,native_language,meeting
0,FEE005,F,20.0,English,ES2002
1,MEE006,M,25.0,English,ES2002
2,MEE007,M,21.0,English,ES2002
3,MEE008,M,27.0,English,ES2002
4,MEE009,M,NaN,English,ES2003
5,MEE010,M,25.0,English,ES2003
6,MEE011,M,29.0,English,ES2003
7,MEE012,M,20.0,English,ES2003
8,FEE013,F,20.0,English,ES2004
9,MEE014,M,18.0,English,ES2004


In [ ]:
tiny_i_df = fetch_results('/root/master_thesis/thesis_multi_speaker_asr/src/results/hpc_results/baseline/tiny_int8_cpu_threads_4_8gb.jsonl')
tiny_f_df = fetch_results('/root/master_thesis/thesis_multi_speaker_asr/src/results/hpc_results/baseline/tiny_float32_cpu_threads_4.jsonl')

